In [2]:
!pip install SPARQLWrapper

from SPARQLWrapper import SPARQLWrapper, JSON

sparql = SPARQLWrapper("http://dbpedia.org/sparql")
sparql.setReturnFormat(JSON)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 9.9 MB/s eta 0:00:00


### Как найти сущность в dbpedia по имени (и типу)?

Простой поиск по тексту с использованием CONTAINS оказался неэффективным для DBpedia:
- Запрос выполняется около 30 секунд
- Для "NoSQL" нашел только Strozzi_NoSQL (конкретную реализацию, а не общее понятие)
- Поиск по Front-end вообще ничего не дал

```
SELECT ?subject ?label
WHERE {
  ?subject rdfs:label ?label .
  FILTER (CONTAINS(LCASE(?label), LCASE("NoSQL")))
}
```

Выводы по поиску:
- Для полнотекстового поиска в DBpedia необходимо использовать bif:contains вместо стандартного FILTER(CONTAINS(...)). Это дает стабильные результаты в рамках лимитов времени выполнения.
- Свойство skos:altLabel теоретически позволяет искать по синонимам, но на практике заполнено редко
- Поиск по классу (a dbo:Profession) работает только если у сущности явно указан этот тип. Многие сущности (например, Front-end web development) не имеют типа или имеют только owl:Thing.

In [3]:
def search_resources(name, class_type=None):
    if class_type:
        type_pattern = f"a {class_type} ;"
    else:
        type_pattern = ""

    query = f"""
    SELECT DISTINCT ?resource ?label WHERE {{
        ?resource {type_pattern} rdfs:label ?label .
        FILTER(lang(?label) = 'en')
        ?label bif:contains '"{name}"'
    }}
    LIMIT 20
    """

    sparql.setQuery(query)
    results = sparql.query().convert()

    print(f"Поиск '{name}' как {class_type if class_type else 'любой тип'}:")
    for r in results["results"]["bindings"]:
        print(f"  {r['label']['value']} - {r['resource']['value']}")

    if not results["results"]["bindings"]:
        print(f"  Ничего не найдено")

search_resources("MLOPS")
print()
search_resources("front-end")
print()
search_resources("NoSQL")
print()
search_resources("python", "dbo:ProgrammingLanguage")
print()
search_resources("Data scientist", "dbo:Profession")
print()
search_resources("Markup languages", "skos:Concept")
print()
search_resources("Machine learning", "dbo:AcademicSubject")

Поиск 'MLOPS' как любой тип:
  MLOps - http://dbpedia.org/resource/MLOps

Поиск 'front-end' как любой тип:
  Front end (computing) - http://dbpedia.org/resource/Front_end_(computing)
  Bootstrap (front-end framework) - http://dbpedia.org/resource/Bootstrap_(front-end_framework)
  Analog front-end - http://dbpedia.org/resource/Analog_front-end
  Front end innovation - http://dbpedia.org/resource/Front_end_innovation
  Front-end engineering - http://dbpedia.org/resource/Front-end_engineering
  Front-end web development - http://dbpedia.org/resource/Front-end_web_development
  Front-end Robotics Enabling Near-term Demonstration - http://dbpedia.org/resource/Front-end_Robotics_Enabling_Near-term_Demonstration
  Front End Loader - http://dbpedia.org/resource/Front_End_Loader
  Front end - http://dbpedia.org/resource/Front_end
  Front end of line - http://dbpedia.org/resource/Front_end_of_line
  Front-end bra - http://dbpedia.org/resource/Front-end_bra
  Front-end loading - http://dbpedia.or

### Поиск связей, которые могут описывать связи для графа компетенций

Для выявления потенциальных зависимостей между навыками был выполнен анализ свойств экземпляров классов dbo:Profession и dbo:AcademicSubject.
1. Функция `explore_class_properties` вывела список всех свойств, используемых у экземпляров этих классов.
2. Из списка были отобраны свойства, которые по смыслу могли указывать на предварительные требования или связи между компетенциями
3. Для каждого отобранного свойства функция `analyze_property` показала, с какими объектами оно связывает сущности

Результат: свойства, явно указывающие на то, что навык/технология требует для изучения другой навык/технологию, отсутствуют. Имеющиеся связи использутся крайне выборочно (отсутсвуют в it-профессиях) и  несут в себе общую информацию.

In [10]:
def explore_class_properties(class_name, min_count=1):
    query = f"""
    SELECT ?property (COUNT(*) AS ?tripleCount) (SAMPLE(?propLabel) AS ?propertyLabel)
    WHERE {{
        ?instance a {class_name} .
        ?instance ?property ?value .
        OPTIONAL {{ ?property rdfs:label ?propLabel . FILTER(lang(?propLabel) = 'en') }}
    }}
    GROUP BY ?property
    HAVING (COUNT(*) >= {min_count})
    ORDER BY DESC(?tripleCount)
    """

    sparql.setQuery(query)
    results = sparql.query().convert()

    print(f"Свойства для класса {class_name} (мин. использований: {min_count}):")
    for r in results["results"]["bindings"]:
        count = r['tripleCount']['value']
        if 'propertyLabel' in r:
            prop_name = r['propertyLabel']['value']
        else:
            prop_name = r['property']['value'].split('/')[-1]
        print(f"  {prop_name} - {r['property']['value']} ({count})")

In [5]:
explore_class_properties("dbo:Profession", 5)

Свойства для класса dbo:Profession (мин. использований: 5):
  Link from a Wikipage to another Wikipage - http://dbpedia.org/ontology/wikiPageWikiLink (170814)
  description - http://dbpedia.org/ontology/description (36576)
  owl#sameAs - http://www.w3.org/2002/07/owl#sameAs (25333)
  wikiPageUsesTemplate - http://dbpedia.org/property/wikiPageUsesTemplate (12793)
  rdf-schema#label - http://www.w3.org/2000/01/rdf-schema#label (11648)
  Link from a Wikipage to an external page - http://dbpedia.org/ontology/wikiPageExternalLink (6730)
  22-rdf-syntax-ns#type - http://www.w3.org/1999/02/22-rdf-syntax-ns#type (5810)
  subject - http://purl.org/dc/terms/subject (5295)
  depiction - http://xmlns.com/foaf/0.1/depiction (4034)
  prov#wasDerivedFrom - http://www.w3.org/ns/prov#wasDerivedFrom (2166)
  isPrimaryTopicOf - http://xmlns.com/foaf/0.1/isPrimaryTopicOf (2165)
  thumbnail - http://dbpedia.org/ontology/thumbnail (1904)
  hypernym - http://purl.org/linguistics/gold/hypernym (1179)
  Wikipa

In [6]:
def analyze_property(property_pattern, subject_class_name=None, limit=None):
    if subject_class_name:
        subject_class_filter = f"?s a {subject_class_name} ."
    else:
        subject_class_filter = ""

    where_block = f"""
    WHERE {{
        ?s {property_pattern} ?o .
        {subject_class_filter}
        ?s rdfs:label ?subjectLabel .
        ?o rdfs:label ?objectLabel .
        FILTER(lang(?subjectLabel) = 'en' && lang(?objectLabel) = 'en')
    }}
    """

    count_query = f"""
    SELECT (COUNT(*) AS ?count)
    {where_block}
    """

    sparql.setQuery(count_query)
    count_result = sparql.query().convert()
    count = count_result["results"]["bindings"][0]["count"]["value"]
    print(f"Количество триплетов с английскими лейблами {property_pattern}: {count}")

    example_query = f"""
    SELECT ?subjectLabel ?objectLabel
    {where_block}
    """

    if limit is not None:
        example_query += f" LIMIT {limit}"

    sparql.setQuery(example_query)
    example_results = sparql.query().convert()

    print(f"\nПримеры:")
    for r in example_results["results"]["bindings"]:
        print(f"  {r['subjectLabel']['value']} → {r['objectLabel']['value']}")

In [7]:
subject_class_name = "dbo:Profession"
analyze_property("dbp:competencies", subject_class_name)
analyze_property("dbp:requirements", subject_class_name)
analyze_property("dbp:specialty", subject_class_name)
analyze_property("dbp:employmentField", subject_class_name)
analyze_property("dbp:activitySector", subject_class_name)
analyze_property("dbp:relatedOccupation", subject_class_name)
analyze_property("dbp:formation", subject_class_name)
analyze_property("gold:hypernym", subject_class_name)

Количество триплетов с английскими лейблами dbp:competencies: 81

Примеры:
  Scientist → Research
  Stripper → Lap dance
  Stripper → Go-go dancing
  Stripper → Striptease
  Stripper → Pole dance
  Store manager → Revenue management
  Store manager → Human resource management
  Store manager → Event management
  Store manager → Marketing
  Store manager → Customer relationship management
  Store manager → Sales management
  Store manager → Operations management
  Store manager → Financial management
  Store manager → Group-dynamic game
  Store manager → Team building
  Businessperson → Social network
  Businessperson → Risk
  Businessperson → Innovation
  Businessperson → Leadership
  Businessperson → Goal
  Businessperson → Critical thinking
  Businessperson → Persuasion
  Poet → Writing
  Structural engineer → Project management
  Structural engineer → Creativity
  Structural engineer → Design
  Structural engineer → Analysis
  Structural engineer → Engineering economics
  Structural

In [8]:
explore_class_properties("dbo:AcademicSubject", 5)

Свойства для класса dbo:AcademicSubject (мин. использований: 5):
  Link from a Wikipage to another Wikipage - http://dbpedia.org/ontology/wikiPageWikiLink (384368)
  owl#sameAs - http://www.w3.org/2002/07/owl#sameAs (48988)
  description - http://dbpedia.org/ontology/description (33962)
  Link from a Wikipage to an external page - http://dbpedia.org/ontology/wikiPageExternalLink (24750)
  wikiPageUsesTemplate - http://dbpedia.org/property/wikiPageUsesTemplate (20776)
  rdf-schema#label - http://www.w3.org/2000/01/rdf-schema#label (18359)
  depiction - http://xmlns.com/foaf/0.1/depiction (7290)
  subject - http://purl.org/dc/terms/subject (5149)
  22-rdf-syntax-ns#type - http://www.w3.org/1999/02/22-rdf-syntax-ns#type (4031)
  thumbnail - http://dbpedia.org/ontology/thumbnail (2004)
  isPrimaryTopicOf - http://xmlns.com/foaf/0.1/isPrimaryTopicOf (1796)
  prov#wasDerivedFrom - http://www.w3.org/ns/prov#wasDerivedFrom (1796)
  hypernym - http://purl.org/linguistics/gold/hypernym (1135)
  

In [ ]:
# 1. Иерархические связи "is-a" / "broader-narrower"
subject_class_name = "dbo:AcademicSubject"
analyze_property("dct:subject", subject_class_name, 100)           # Темы/категории предмета
analyze_property("gold:hypernym", subject_class_name, 100)         # Родовые понятия
analyze_property("skos:broaderMatch", subject_class_name, 100)     # Более широкие понятия
analyze_property("skos:narrower", subject_class_name, 100)         # Более узкие понятия
analyze_property("dbo:wikiPageWikiLink", subject_class_name, 100)  # Связанные страницы

# 2. Связи "часть-целое" и специализация
analyze_property("dbo:subdivision", subject_class_name)       # Разделы дисциплины
analyze_property("dbo:specialization", subject_class_name)    # Специализации
analyze_property("dbo:focus", subject_class_name)             # Фокус/направление

# 3. Образование и ресурсы
analyze_property("dbp:formation", subject_class_name)         # Образовательные программы
analyze_property("dbp:competencies", subject_class_name)      # Требуемые компетенции
analyze_property("foaf:homepage", subject_class_name)         # Внешние ресурсы

# 4. Контекстные связи
analyze_property("dbp:activitySector", subject_class_name)    # Сектор деятельности
analyze_property("dbp:employmentField", subject_class_name)   # Область занятости
analyze_property("dbp:relatedOccupation", subject_class_name) # Связанные профессии
analyze_property("dbo:wikiPageRedirects", subject_class_name) # Синонимы/перенаправления

# 5. Точные соответствия (для объединения данных)
analyze_property("skos:exactMatch", subject_class_name)       # Точные соответствия
analyze_property("skos:closeMatch", subject_class_name)       # Близкие соответствия
analyze_property("owl:sameAs", subject_class_name)            # Те же сущности в других базах

Количество триплетов с английскими лейблами dct:subject: 5431

Примеры:
  Telematics → Telematics
  Educational assessment → Evaluation methods
  Molinology → History of technology
  Hospitality management studies → Education by subject
  Geopoetics → Poetry
  Strategic communication → Communication
  Land change science → Earth sciences
  Film studies → Film
  Humanities → Society
  Traditional ecological knowledge → Ecology
  Mathematical statistics → Statistical theory
  Holocaust studies → The Holocaust
  Citizen science → Crowdsourcing
  Telematics → American inventions
  Economics of language → Cultural economics
  Bachelor of Letters → Education in England
  Forest gardening → Agriculture and the environment
  Agroforestry → Agriculture and the environment
  Conservation biology → Environmental conservation
  Environmental epidemiology → Environmental health
  Practical theology → Practical theology
  Anthropometry → Human anatomy
  Fluid mechanics → Fluid mechanics
  Electric p

### Определение класса сущности

Так как поиск конкретных связей успехом не увенчался, я поставил задачу научиться определять классы сущностей.

#### Как можно было бы строить связи через wikiPageWikiLink


Если классификация сущности известна (например, установлено, что сущность относится к классу dbo:ProgrammingLanguage или dbo:Software), можно использовать свойство dbo:wikiPageWikiLink для извлечения связанных сущностей.

Например так: из всех wikiPageWikiLink сущности отбираются те, у которых целевая сущность также имеет определенный класс (например, для языка программирования — связанные фреймворки, библиотеки, инструменты)

#### Проблемы автоматической классификации сущностей:
- Отсутствие типа: многие сущности (например, Front-end web development) не имеют явного rdf:type или имеют только owl:Thing, что не дает информации о природе сущности.
- Категории как основной источник: для многих сущностей единственная содержательная связь — это dct:subject, указывающий на категорию (dbc:). Однако категорий слишком много, чтобы хардкодить их отбор.
- Нет единой иерархии: одна сущность может принадлежать множеству категорий разного уровня обобщения (например, Statistics относится и к dbc:Statistics, и к dbc:Arab_inventions)

#### Категории как иерархическая структура
Категории в DBpedia (dbc:) представлены как экземпляры skos:Concept. Между категориями заданы иерархические связи через skos:broader и skos:narrower.

Это позволяет строить иерархию знаний, где более общие понятия предшествуют узким. Однако, как указано в спецификации [SKOS Reference](https://www.w3.org/TR/skos-reference/) (раздел 1.3), такие структуры не имеют формальной семантики и не могут интерпретироваться как аксиомы:

> These structures, however, do not have any formal semantics, and cannot be reliably interpreted as either formal axioms or facts about the world.


#### Запрос, который использовался для анализа классов сущностей
```
SELECT DISTINCT ?property ?value ?valueLabel WHERE {
  BIND(<http://dbpedia.org/resource/Front-end_web_development> AS ?entity)
  
  {
    ?entity a ?class .
    ?class rdfs:label ?valueLabel .
    BIND("тип (класс)" AS ?property)
    BIND(?class AS ?value)
    FILTER(LANG(?valueLabel) = 'en')
  }
  UNION
  {
    ?entity gold:hypernym ?value .
    ?value rdfs:label ?valueLabel .
    FILTER(LANG(?valueLabel) = 'en')
    BIND("гипероним" AS ?property)
  }
  UNION
  {
    ?entity dct:subject ?value .
    ?value rdfs:label ?valueLabel .
    FILTER(LANG(?valueLabel) = 'en')
    BIND("категория" AS ?property)
  }
  UNION
  {
    ?entity rdfs:subClassOf ?value .
    ?value rdfs:label ?valueLabel .
    FILTER(LANG(?valueLabel) = 'en')
    BIND("подкласс" AS ?property)
  }
  UNION
  {
    ?entity dbo:genre ?value .
    ?value rdfs:label ?valueLabel .
    FILTER(LANG(?valueLabel) = 'en')
    BIND("жанр" AS ?property)
  }
  UNION
  {
    ?entity dbp:type ?value .
    ?value rdfs:label ?valueLabel .
    FILTER(LANG(?valueLabel) = 'en')
    BIND("тип (инфобокс)" AS ?property)
  }
  UNION
  {
    ?entity dbp:category ?value .
    ?value rdfs:label ?valueLabel .
    FILTER(LANG(?valueLabel) = 'en')
    BIND("категория (инфобокс)" AS ?property)
  }
  UNION
  {
    ?entity dbp:classification ?value .
    ?value rdfs:label ?valueLabel .
    FILTER(LANG(?valueLabel) = 'en')
    BIND("классификация" AS ?property)
  }
}
```